## Лабораторная работа №3. Обучение с учителем. Деревья решений и их ансамбли.

### (1) IEEE-CIS Fraud Detection — задача бинарной классификации мошеннических онлайн-транзакций.

### 1. Импорт библиотек

Нам понадобятся библиотеки для работы с файловой системой ВМ, загрузки, просмотра и обработки датасетов, а также обучения модели классификации.

In [14]:
# работа с файловой системой
import os

# чистка traceback
import warnings
warnings.filterwarnings(
    "ignore",
    category=FutureWarning
)
warnings.filterwarnings(
    "ignore",
    category=UserWarning
)

# работа с датасетами как с массивами и таблицами
import numpy as np
import pandas as pd

# предварительная обработка данных в датасетах
from sklearn.base import BaseEstimator, TransformerMixin, ClassifierMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from feature_engine.encoding import CountFrequencyEncoder

# пользовательские трансформеры
from custom_transformers import OutlierClipper, CyclicalTimeEncoder

# написание своих деревьев решений
from dataclasses import dataclass
from typing import Optional, Union

# параллельное обучение деревьев
from joblib import Parallel, delayed

# готовые деревья решений из sklearn
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier as SkBaggingClassifier, RandomForestClassifier as SkRandomForestClassifier

# готовый градиентный бустинг из sklearn, XGBoost, LightGBM и CatBoost
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# оценка точности предсказаний модели
from sklearn.metrics import accuracy_score, f1_score, classification_report

# перенос пайплайна fraud_pipeline.joblib и колонок fraud_processed_columns.joblib из ЛР1
import joblib

# запись результатов экспериментов в Google Sheet
import json
import gspread
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials

### 2. Загрузка и применение пайплайна предобработки данных из ЛР1

Сначала узнаем, как называются файлы в датасете, найдем нужный нам файл с данными для обучения и посмотрим, как он выглядит в общих чертах.

In [2]:
# путь к датасету согласно документации Yandex Cloud
dataset_path = "/home/jupyter/datasets/ieee-fraud-detection"

# список всех файлов в датасете
print("Файлы в датасете:")
print(os.listdir(dataset_path))

# путь к train_public
file_path = os.path.join(dataset_path, "train_public.csv")
# загрузка csv-файла
df = pd.read_csv(file_path)

# первые 5 строк
print("\nНабор тренировочных данных:")
df.head()

Файлы в датасете:
['train_public.csv', 'test_public.csv', 'sample_submission.csv']

Набор тренировочных данных:


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,card6,addr1,addr2,dist1,dist2,P_emaildomain,R_emaildomain,C1,C2,C3,C4,C5,C6,C7,C8,C9,C10,C11,C12,C13,C14,D1,D2,D3,D4,D5,D6,D7,D8,D9,...,id_01,id_02,id_03,id_04,id_05,id_06,id_07,id_08,id_09,id_10,id_11,id_12,id_13,id_14,id_15,id_16,id_17,id_18,id_19,id_20,id_21,id_22,id_23,id_24,id_25,id_26,id_27,id_28,id_29,id_30,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,credit,315.0,87.0,19.0,NaN,NaN,NaN,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,2.0,0.0,1.0,1.0,14.0,NaN,13.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,credit,325.0,87.0,NaN,NaN,gmail.com,NaN,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,debit,330.0,87.0,287.0,NaN,outlook.com,NaN,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,debit,476.0,87.0,NaN,NaN,yahoo.com,NaN,2.0,5.0,0.0,0.0,0.0,4.0,0.0,0.0,1.0,0.0,1.0,0.0,25.0,1.0,112.0,112.0,0.0,94.0,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,credit,420.0,87.0,NaN,NaN,gmail.com,NaN,1.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,70787.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.0,NotFound,NaN,-480.0,New,NotFound,166.0,NaN,542.0,144.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,New,NotFound,Android 7.0,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M


Разделим train_public на данные для обучения и для тестирования модели машинного обучения. Обработаем получившиеся два массива данных по отдельности пайплайном из ЛР1.

In [3]:
# постоянный сид для воспроизводимости результатов
seed = 8

# разделение train_public на признаки и целевую переменную isFraud
X = df.drop(columns=["isFraud"])
y = df["isFraud"]

# разделение train_public на train data и test data
X_tr, X_te, y_tr, y_te = train_test_split(
    X,
    y,
    test_size=0.1,
    random_state=seed,
    stratify=y  # для несбалансированных данных в ieee-fraud-detection
)

# загрузка пайплайна и имен колонок
pipeline = joblib.load("fraud_pipeline.joblib")
processed_columns = joblib.load("fraud_processed_columns.joblib")

# предобработка данных пайплайном
X_tr_processed = pipeline.transform(X_tr)
X_te_processed = pipeline.transform(X_te)

### 3. Реализация BaggingClassifier и RandomForestClassifier в виде sklearn-совместимых классов

Базовый алгоритм для ансамблей - бинарное дерево решений на критерии энтропии Шеннона.

In [4]:
# Узел дерева решений. Может быть либо листом, либо внутренним узлом.
@dataclass
class _ClassificationNode:
    is_leaf: bool  # флаг: является ли узел листом
    prediction: Optional[int] = None  # предсказанный класс (для листа)
    proba: Optional[np.ndarray] = None  # вероятности классов (для листа)
    feature_index: Optional[int] = None  # индекс признака для сплита (для внутреннего узла)
    threshold: Optional[float] = None  # порог разбиения
    left: Optional["_ClassificationNode"] = None  # левый потомок (<= threshold)
    right: Optional["_ClassificationNode"] = None  # правый потомок (> threshold)

# Простая реализация дерева решений для бинарной классификации
class SimpleDecisionTreeClassifier(BaseEstimator, ClassifierMixin):

    def __init__(
    self,
    max_depth: Optional[int] = None,
    min_samples_split: int = 2,
    min_samples_leaf: int = 1,
    min_impurity_decrease: float = 0.0,
    max_thresholds: int = 512,
    max_features: Optional[Union[int, float, None]] = None,
    random_state: Optional[int] = None):
        # сохраняем гиперпараметры модели
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.min_impurity_decrease = min_impurity_decrease
        self.max_thresholds = max_thresholds
        self.max_features = max_features
        self.random_state = random_state
    
    # Считает энтропию Шеннона для массива меток y
    def _entropy(self, y: np.ndarray) -> float:
        # используется как мера классовой "нечистоты" узла
        if y.size == 0:
            return 0.0  # пустой набор → нулевая энтропия

        # считаем количество объектов каждого класса
        counts = np.bincount(y, minlength=self.n_classes_).astype(float)
        probs = counts / counts.sum()  # переводим в вероятности

        # убираем нули, чтобы избежать log2(0)
        probs = probs[probs > 0]
        return -np.sum(probs * np.log2(probs))

    # Создаёт листовой узел: считает вероятности классов и предсказание
    def _leaf(self, y: np.ndarray) -> _ClassificationNode:
        counts = np.bincount(y, minlength=self.n_classes_).astype(float)
        proba = counts / counts.sum()  # распределение классов
        prediction = int(np.argmax(proba))  # самый вероятный класс

        return _ClassificationNode(is_leaf=True, prediction=prediction, proba=proba)

    # Определяет, сколько признаков использовать при поиске сплита
    def _resolve_max_features(self, n_features: int) -> int:
        mf = self.max_features
        if mf is None:
            return n_features  # используем все признаки
        if isinstance(mf, int):
            if mf <= 0:
                raise ValueError("max_features as int must be >= 1.")
            return min(mf, n_features)  # не больше общего числа
        if isinstance(mf, float):
            if not (0.0 < mf <= 1.0):
                raise ValueError("max_features as float must be in (0, 1].")
            return max(1, int(np.ceil(mf * n_features))) # берём долю признаков
        raise ValueError("Unsupported type for max_features.")

    # Выбирает случайное подмножество признаков для сплита
    def _feature_indices_for_split(self, n_features: int) -> np.ndarray:
        k = self._resolve_max_features(n_features)

        if k == n_features:
            return np.arange(n_features)  # все признаки

        # случайная выборка без возвращения
        return self._rng.choice(n_features, size=k, replace=False)

    # Ищет лучший сплит по всем выбранным признакам и порогам
    def _best_split(self, X: np.ndarray, y: np.ndarray):

        n_samples, n_features = X.shape
        parent_impurity = self._entropy(y)  # энтропия родительского узла
        best = None  # сюда запишем лучший вариант (impurity, feature, threshold)

        feature_indices = self._feature_indices_for_split(n_features)

        # перебираем признаки
        for j in feature_indices:
            xj = X[:, j]

            # сортируем по возрастанию значения признака и соответствующие метки
            order = np.argsort(xj)
            x_sorted = xj[order]
            y_sorted = y[order]

            uniq = np.unique(x_sorted)
            if uniq.size < 2:
                continue  # нет смысла делить, все значения одинаковые

            # генерация кандидатов порогов
            if uniq.size > self.max_thresholds + 1:
                # слишком много уникальных значений → слишком много порогов, ограничиваем их числом max_thresholds порогов
                qs = np.linspace(0, 1, self.max_thresholds + 2)[1:-1] # равномерное разбиение на доли единицы ограниченным числом порогов
                thresholds = np.quantile(x_sorted, qs) # в качестве порогов берём значения признака на полученных выше квантилях
                thresholds = np.unique(thresholds) # убираем дубликаты
            else:
                # в качестве порогов берём середины между соседними значениями
                thresholds = (uniq[:-1] + uniq[1:]) / 2.0

            # проверяем каждый порог
            for t in thresholds:
                left_mask = x_sorted <= t
                right_mask = ~left_mask

                n_left = int(left_mask.sum())
                n_right = n_samples - n_left

                # проверка ограничения на минимальный размер листа
                if n_left < self.min_samples_leaf or n_right < self.min_samples_leaf:
                    continue

                y_left = y_sorted[left_mask]
                y_right = y_sorted[right_mask]

                # считаем энтропии детей
                h_left = self._entropy(y_left)
                h_right = self._entropy(y_right)

                # взвешенная энтропия после разбиения
                weighted_impurity = (
                    (n_left / n_samples) * h_left +
                    (n_right / n_samples) * h_right
                )

                # сохраняем лучший (минимальная impurity)
                if best is None or weighted_impurity < best[0]:
                    best = (weighted_impurity, j, t)

        return best, parent_impurity

    # Проверяет условия остановки роста дерева
    def _should_stop(self, X: np.ndarray, y: np.ndarray, depth: int) -> bool:
        if self.max_depth is not None and depth >= self.max_depth:
            return True  # достигли максимальной глубины
        if X.shape[0] < self.min_samples_split:
            return True  # слишком мало объектов
        if np.unique(y).size == 1:
            return True  # все объекты одного класса
        return False

    # Рекурсивно строит дерево
    def _build(self, X: np.ndarray, y: np.ndarray, depth: int) -> _ClassificationNode:

        if self._should_stop(X, y, depth):
            return self._leaf(y)

        best, parent_impurity = self._best_split(X, y)

        if best is None:
            return self._leaf(y)  # не нашли хороший сплит

        child_impurity, j, t = best
        improvement = parent_impurity - child_impurity

        # проверка минимального улучшения
        if improvement < self.min_impurity_decrease: # если улучшение разбиения слишком маленькое, то не делаем сплит
            return self._leaf(y)

        # делим данные
        left_mask = X[:, j] <= t
        right_mask = ~left_mask

        # рекурсивно строим поддеревья
        left = self._build(X[left_mask], y[left_mask], depth + 1)
        right = self._build(X[right_mask], y[right_mask], depth + 1)

        # создаём внутренний узел
        return _ClassificationNode(
            is_leaf=False,
            feature_index=j,
            threshold=t,
            left=left,
            right=right,
        )
    
    # Обучение дерева
    def fit(self, X, y):

        X = np.asarray(X)
        y = np.asarray(y)

        # кодируем классы в числа (0 и 1)
        self.classes_, y_encoded = np.unique(y, return_inverse=True)

        # ограничение: только бинарная классификация
        if self.classes_.shape[0] != 2:
            raise ValueError("This implementation supports only binary classification.")

        self.n_classes_ = len(self.classes_)
        self.n_features_in_ = X.shape[1]

        # генератор случайных чисел
        self._rng = np.random.RandomState(self.random_state)

        # строим дерево
        self.root_ = self._build(X, y_encoded, depth=0)

        return self

    # Проходит по дереву от корня до листа для одного объекта
    def _predict_proba_row(self, node: _ClassificationNode, x: np.ndarray) -> np.ndarray:
        while not node.is_leaf:
            if x[node.feature_index] <= node.threshold:
                node = node.left
            else:
                node = node.right
        return node.proba  # возвращаем вероятности из листа

    # Предсказание вероятностей для всех объектов
    def predict_proba(self, X):
        X = np.asarray(X)
        proba = np.array([self._predict_proba_row(self.root_, x) for x in X])
        return proba

    # Предсказание классов (argmax по вероятностям)
    def predict(self, X):
        proba = self.predict_proba(X)
        pred_idx = np.argmax(proba, axis=1)
        return self.classes_[pred_idx]

Bagging Classifier - ансамбль бинарных решающих деревьев на бутстрапе.

In [5]:
# Ансамбль деревьев решений (bagging)
# Каждое дерево обучается на случайной подвыборке, итог - усреднение предсказаний
class BaggingClassifier(BaseEstimator, ClassifierMixin):

    def __init__(
        self,
        n_estimators: int = 10,
        max_depth: Optional[int] = None,
        min_samples_split: int = 2,
        min_samples_leaf: int = 1,
        min_impurity_decrease: float = 0.0,
        max_thresholds: int = 512,
        bootstrap: bool = True,
        max_samples: Union[int, float] = 1.0,
        n_jobs: int = 1,
        random_state: Optional[int] = None,
    ):
        # сохраняем гиперпараметры ансамбля и базового дерева
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.min_impurity_decrease = min_impurity_decrease
        self.max_thresholds = max_thresholds
        self.bootstrap = bootstrap
        self.max_samples = max_samples
        self.n_jobs = n_jobs
        self.random_state = random_state

    # Определяет размер подвыборки:
    def _resolve_sample_size(self, n_samples: int) -> int:
        if isinstance(self.max_samples, float):
            if not (0.0 < self.max_samples <= 1.0):
                raise ValueError("max_samples as float must be in (0, 1].")
            return max(1, int(self.max_samples * n_samples)) # берём долю объектов
        if isinstance(self.max_samples, int):
            if self.max_samples <= 0:
                raise ValueError("max_samples as int must be >= 1.")
            return min(self.max_samples, n_samples) # не больше общего числа объектов
        raise ValueError("max_samples must be int or float.")

    # Обучает одно дерево на случайной подвыборке
    def _fit_single_estimator(self, X: np.ndarray, y: np.ndarray, seed: int):

        rng = np.random.RandomState(seed)
        n_samples = X.shape[0]
        sample_size = self._resolve_sample_size(n_samples)

        # случайная выборка индексов (с повторением или без)
        indices = rng.choice(
            n_samples,
            size=sample_size,
            replace=self.bootstrap
        )

        X_boot = X[indices]
        y_boot = y[indices]

        # базовый алгоритм - обычное дерево
        tree = SimpleDecisionTreeClassifier(
            max_depth=self.max_depth,
            min_samples_split=self.min_samples_split,
            min_samples_leaf=self.min_samples_leaf,
            min_impurity_decrease=self.min_impurity_decrease,
            max_thresholds=self.max_thresholds,
            max_features=None,  # в bagging используем все признаки
            random_state=seed,
        )

        tree.fit(X_boot, y_boot)
        return tree

    # Обучает ансамбль
    def fit(self, X, y):

        X = np.asarray(X)
        y = np.asarray(y)

        # сохраняем исходные классы
        self.classes_, _ = np.unique(y, return_inverse=True)

        # ограничение: только бинарная классификация
        if self.classes_.shape[0] != 2:
            raise ValueError("This implementation supports only binary classification.")

        self.n_features_in_ = X.shape[1]

        # отдельный seed для каждого дерева
        rng = np.random.RandomState(self.random_state)
        seeds = rng.randint(0, 10**9, size=self.n_estimators)

        # параллельное обучение
        self.estimators_ = Parallel(n_jobs=self.n_jobs)(
            delayed(self._fit_single_estimator)(X, y, seed)
            for seed in seeds
        )

        return self

    # Каждое дерево даёт вероятности → усредняем
    def predict_proba(self, X):
        X = np.asarray(X)
        all_probas = np.array([
            est.predict_proba(X)
            for est in self.estimators_
        ])
        return np.mean(all_probas, axis=0)

    # Предсказание классов (по вероятности положительного)
    def predict(self, X, threshold: float = 0.5):
        proba_pos = self.predict_proba(X)[:, 1]
        pred = (proba_pos >= threshold).astype(int)
        return self.classes_[pred]

Random Forest Classifier - ансамбль бинарных решающих деревьев на бутстрапе c выбором n из всех N признаков.

In [6]:
# Ансамбль деревьев решений (random forest)
# Каждое дерево обучается на случайной подвыборке объектов, а при каждом сплите рассматривает только случайное подмножество признаков
class RandomForestClassifier(BaseEstimator, ClassifierMixin):
 
    def __init__(
        self,
        n_estimators: int = 10,
        max_depth: Optional[int] = None,
        min_samples_split: int = 2,
        min_samples_leaf: int = 1,
        min_impurity_decrease: float = 0.0,
        max_thresholds: int = 512,
        max_features: Optional[Union[int, float, None]] = 0.5,
        bootstrap: bool = True,
        max_samples: Union[int, float] = 1.0,
        n_jobs: int = 1,
        random_state: Optional[int] = None,
    ):
        # сохраняем гиперпараметры ансамбля и базового дерева
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.min_impurity_decrease = min_impurity_decrease
        self.max_thresholds = max_thresholds
        self.max_features = max_features
        self.bootstrap = bootstrap
        self.max_samples = max_samples
        self.n_jobs = n_jobs
        self.random_state = random_state
 
    # Определяет размер подвыборки
    def _resolve_sample_size(self, n_samples: int) -> int:
        if isinstance(self.max_samples, float):
            if not (0.0 < self.max_samples <= 1.0):
                raise ValueError("max_samples as float must be in (0, 1].")
            return max(1, int(self.max_samples * n_samples))  # берём долю объектов
        if isinstance(self.max_samples, int):
            if self.max_samples <= 0:
                raise ValueError("max_samples as int must be >= 1.")
            return min(self.max_samples, n_samples)  # не больше общего числа объектов
 
        raise ValueError("max_samples must be int or float.")
 
    # Обучает одно дерево на случайной подвыборке
    def _fit_single_estimator(self, X: np.ndarray, y: np.ndarray, seed: int):
        rng = np.random.RandomState(seed)
        n_samples = X.shape[0]
        sample_size = self._resolve_sample_size(n_samples)
 
        # случайная выборка индексов (с повторением или без)
        indices = rng.choice(
            n_samples,
            size=sample_size,
            replace=self.bootstrap
        )
 
        X_boot = X[indices]
        y_boot = y[indices]
 
        # базовый алгоритм - дерево со случайным подмножеством признаков при сплитах
        tree = SimpleDecisionTreeClassifier(
            max_depth=self.max_depth,
            min_samples_split=self.min_samples_split,
            min_samples_leaf=self.min_samples_leaf,
            min_impurity_decrease=self.min_impurity_decrease,
            max_thresholds=self.max_thresholds,
            max_features=self.max_features,  # главное отличие random forest от bagging
            random_state=seed,
        )
 
        tree.fit(X_boot, y_boot)
        return tree
 
    # Обучает ансамбль
    def fit(self, X, y):
        
        X = np.asarray(X)
        y = np.asarray(y)
 
        # сохраняем исходные классы
        self.classes_, _ = np.unique(y, return_inverse=True)
 
        # ограничение: только бинарная классификация
        if self.classes_.shape[0] != 2:
            raise ValueError("This implementation supports only binary classification.")
 
        self.n_features_in_ = X.shape[1]
 
        # отдельный seed для каждого дерева
        rng = np.random.RandomState(self.random_state)
        seeds = rng.randint(0, 10**9, size=self.n_estimators)
 
        # параллельное обучение
        self.estimators_ = Parallel(n_jobs=self.n_jobs)(
            delayed(self._fit_single_estimator)(X, y, seed)
            for seed in seeds
        )
 
        return self
 
    # Каждое дерево даёт вероятности → усредняем
    def predict_proba(self, X):
        X = np.asarray(X)
        all_probas = np.array([
            est.predict_proba(X)
            for est in self.estimators_
        ])
        return np.mean(all_probas, axis=0)
 
    # Предсказание классов (по вероятности положительного)
    def predict(self, X, threshold: float = 0.5):
        proba_pos = self.predict_proba(X)[:, 1]
        pred = (proba_pos >= threshold).astype(int)
        return self.classes_[pred]

Для корректировки параметров сначала оценим качество предсказаний одного дерева.

In [7]:
sdt = SimpleDecisionTreeClassifier(max_depth=8, min_samples_split=30, min_samples_leaf=15, max_thresholds=512, random_state=seed).fit(X_tr_processed, y_tr)
y_pred_sdt = sdt.predict(X_te_processed)

print("accuracy:", accuracy_score(y_te, y_pred_sdt))
print(classification_report(y_te, y_pred_sdt, target_names=["not fraud", "fraud"]))

accuracy: 0.9740707814748962
              precision    recall  f1-score   support

   not fraud       0.98      1.00      0.99     45584
       fraud       0.83      0.33      0.47      1660

    accuracy                           0.97     47244
   macro avg       0.90      0.66      0.73     47244
weighted avg       0.97      0.97      0.97     47244



Даже у одного решающего дерева с умеренными параметрами для fraud f1-score = 0.47, что уже выше, чем у линейных моделей с их лучшими порогами.

С теми же параметрами обучим наш Bagging Classifier на train data, предскажем значения для test data и оценим результаты.

In [8]:
bc = BaggingClassifier(n_estimators=5, max_depth=8, min_samples_split=30, min_samples_leaf=15, max_thresholds=512, n_jobs=32, random_state=seed).fit(X_tr_processed, y_tr)
y_pred_bc = bc.predict(X_te_processed)

print("accuracy:", accuracy_score(y_te, y_pred_bc))
print(classification_report(y_te, y_pred_bc, target_names=["not fraud", "fraud"]))

accuracy: 0.9749597832528999
              precision    recall  f1-score   support

   not fraud       0.98      1.00      0.99     45584
       fraud       0.88      0.33      0.48      1660

    accuracy                           0.97     47244
   macro avg       0.93      0.67      0.74     47244
weighted avg       0.97      0.97      0.97     47244



У ансамбля из пяти решающих деревьев результаты чуть лучше, чем у одного дерева - f1-score = 0.48 для fraud за счет увеличения precision.

С теми же параметрами обучим наш Random Forest Classifier на train data, предскажем значения для test data и оценим результаты.

In [9]:
rfc = RandomForestClassifier(n_estimators=5, max_depth=8, min_samples_split=30, min_samples_leaf=15, max_thresholds=512, max_features=0.15, n_jobs=32, random_state=seed).fit(X_tr_processed, y_tr)
y_pred_rfc = rfc.predict(X_te_processed)

print("accuracy:", accuracy_score(y_te, y_pred_rfc))
print(classification_report(y_te, y_pred_rfc, target_names=["not fraud", "fraud"]))

accuracy: 0.9740496147658962
              precision    recall  f1-score   support

   not fraud       0.97      1.00      0.99     45584
       fraud       0.90      0.29      0.44      1660

    accuracy                           0.97     47244
   macro avg       0.94      0.65      0.71     47244
weighted avg       0.97      0.97      0.97     47244



Вырезание признаков в алгоритме Random Forest увеличило precision, но уменьшило recall.

### 4. Сравнение собственных реализаций с готовыми реализациями ансамблевых алгоритмов из sklearn

С теми же параметрами обучим готовый Bagging Classifier из sklearn на train data, предскажем значения для test data и оценим результаты.

In [15]:
bc_sk = SkBaggingClassifier(DecisionTreeClassifier(criterion='entropy', max_depth=8, min_samples_split=30, min_samples_leaf=15, random_state=seed),
                          n_estimators=5, n_jobs=32, random_state=seed).fit(X_tr_processed, y_tr)
y_pred_bc_sk = bc_sk.predict(X_te_processed)

print("accuracy:", accuracy_score(y_te, y_pred_bc_sk))
print(classification_report(y_te, y_pred_bc_sk, target_names=["not fraud", "fraud"]))

accuracy: 0.9746846160358987
              precision    recall  f1-score   support

   not fraud       0.98      1.00      0.99     45584
       fraud       0.90      0.32      0.47      1660

    accuracy                           0.97     47244
   macro avg       0.94      0.66      0.73     47244
weighted avg       0.97      0.97      0.97     47244



С теми же параметрами обучим готовый Random Forest Classifier из sklearn на train data, предскажем значения для test data и оценим результаты.

In [16]:
rfc_sk = SkRandomForestClassifier(n_estimators=5, criterion='entropy', max_depth=8, min_samples_split=30, min_samples_leaf=15, max_features=65, n_jobs=32, random_state=seed).fit(X_tr_processed, y_tr)
y_pred_rfc_sk = rfc_sk.predict(X_te_processed)

print("accuracy:", accuracy_score(y_te, y_pred_rfc_sk))
print(classification_report(y_te, y_pred_rfc_sk, target_names=["not fraud", "fraud"]))

accuracy: 0.9737321141308949
              precision    recall  f1-score   support

   not fraud       0.97      1.00      0.99     45584
       fraud       0.90      0.28      0.43      1660

    accuracy                           0.97     47244
   macro avg       0.94      0.64      0.71     47244
weighted avg       0.97      0.97      0.97     47244



Готовые реализации ансамблевых алгоритмов из sklearn дают результаты лишь на сотую долю хуже, чем собственные реализации, но при этом обучаются кратно быстрее.

### 5. Серия экспериментов с готовыми реализациями градиентного бустинга из sklearn, XGBoost, LightGBM и CatBoost

Посмотрим на результаты четырех различных реализаций градиентного бустинга и сравним их с результатами бэггинга и случайного леса.<br>
Будем подбирать следующие гиперпараметры: n_estimators - количество деревьев; max_depth - глубину одного дерева; min_samples_split вместе с min_samples_leaf - минимальное число объектов для разбиения и соотвественно минимальное число объектов в листе. Чтобы не усложнять эксперимент, пусть min_samples_split всегда в два раза больше min_samples_leaf.

Обучим готовый Gradient Boosting Classifier из sklearn на train data, предскажем значения для test data и оценим результаты.

In [17]:
MODEL_NAME = "SkGradientBoostingClassifier"

if "all_experiment_results" not in globals():
    all_experiment_results = []

def collect_metrics(y_true, y_pred, model_name, params):
    report = classification_report(
        y_true,
        y_pred,
        target_names=["not fraud", "fraud"],
        output_dict=True,
        zero_division=0
    )
    
    return {
        "model": model_name,
        "n_estimators": params["n_estimators"],
        "max_depth": params["max_depth"],
        "min_samples_leaf": params["min_samples_leaf"],
        "min_samples_split": params["min_samples_split"],
        "accuracy": accuracy_score(y_true, y_pred),
        "not fraud precision": report["not fraud"]["precision"],
        "not fraud recall": report["not fraud"]["recall"],
        "not fraud f1-score": report["not fraud"]["f1-score"],
        "fraud precision": report["fraud"]["precision"],
        "fraud recall": report["fraud"]["recall"],
        "fraud f1-score": report["fraud"]["f1-score"],
    }

gb_param_sets = [
    {"n_estimators": 5, "max_depth": 8, "min_samples_split": 30, "min_samples_leaf": 15},
    {"n_estimators": 50, "max_depth": 5, "min_samples_split": 40, "min_samples_leaf": 20},
    {"n_estimators": 100, "max_depth": 3, "min_samples_split": 60, "min_samples_leaf": 30},
]

gb_results = []

for i, params in enumerate(gb_param_sets, start=1):
    model = GradientBoostingClassifier(
        n_estimators=params["n_estimators"],
        max_depth=params["max_depth"],
        min_samples_split=params["min_samples_split"],
        min_samples_leaf=params["min_samples_leaf"],
        random_state=seed,
        verbose=1
    )
    
    model.fit(X_tr_processed, y_tr)
    y_pred = model.predict(X_te_processed)
    
    row = collect_metrics(
        y_true=y_te,
        y_pred=y_pred,
        model_name=MODEL_NAME,
        params=params
    )
    gb_results.append(row)

all_experiment_results = [r for r in all_experiment_results if r["model"] != MODEL_NAME]
all_experiment_results.extend(gb_results)

df_gb_results = pd.DataFrame(gb_results)
display(df_gb_results)

      Iter       Train Loss   Remaining Time 
         1           0.2495            2.31m
         2           0.2374            1.73m
         3           0.2299            1.15m
         4           0.2241           34.59s
         5           0.2172            0.00s
      Iter       Train Loss   Remaining Time 
         1           0.2613           17.77m
         2           0.2531           17.48m
         3           0.2449           17.09m
         4           0.2388           16.75m
         5           0.2328           16.38m
         6           0.2292           16.04m
         7           0.2258           15.67m
         8           0.2225           15.24m
         9           0.2194           14.86m
        10           0.2167           14.50m
        20           0.1989           10.82m
        30           0.1885            7.19m
        40           0.1823            3.60m
        50           0.1778            0.00s
      Iter       Train Loss   Remaining Time 
       

,model,n_estimators,max_depth,min_samples_leaf,min_samples_split,accuracy,not fraud precision,not fraud recall,not fraud f1-score,fraud precision,fraud recall,fraud f1-score
0,SkGradientBoostingClassifier,5,8,15,30,0.972758,0.972885,0.999627,0.986075,0.958231,0.234940,0.377358
1,SkGradientBoostingClassifier,50,5,20,40,0.975743,0.976741,0.998640,0.987569,0.902821,0.346988,0.501305
2,SkGradientBoostingClassifier,100,3,30,60,0.974981,0.976212,0.998399,0.987181,0.883013,0.331928,0.482487


Хорошие результаты (fraud f1-score за 0.5) требуют длительного времени обучения - в районе 20 минут на один большой ансамбль из слабых деревьев.

Обучим готовый Gradient Boosting Classifier из xgboost на train data, предскажем значения для test data и оценим результаты.

In [18]:
MODEL_NAME = "XGBClassifier"

if "all_experiment_results" not in globals():
    all_experiment_results = []

def collect_metrics(y_true, y_pred, model_name, params):
    report = classification_report(
        y_true,
        y_pred,
        target_names=["not fraud", "fraud"],
        output_dict=True,
        zero_division=0
    )
    
    return {
        "model": model_name,
        "n_estimators": params["n_estimators"],
        "max_depth": params["max_depth"],
        "min_samples_split": params["min_samples_split"],
        "min_samples_leaf": params["min_samples_leaf"],
        "accuracy": accuracy_score(y_true, y_pred),
        "not fraud precision": report["not fraud"]["precision"],
        "not fraud recall": report["not fraud"]["recall"],
        "not fraud f1-score": report["not fraud"]["f1-score"],
        "fraud precision": report["fraud"]["precision"],
        "fraud recall": report["fraud"]["recall"],
        "fraud f1-score": report["fraud"]["f1-score"],
    }

# Для XGBoost ближайший аналог min_samples_leaf — min_child_weight
xgb_param_sets = [
    {"n_estimators": 5, "max_depth": 8, "min_samples_split": 30, "min_samples_leaf": 15},
    {"n_estimators": 50, "max_depth": 5, "min_samples_split": 40, "min_samples_leaf": 20},
    {"n_estimators": 100, "max_depth": 3, "min_samples_split": 60, "min_samples_leaf": 30},
]

xgb_results = []

for i, params in enumerate(xgb_param_sets, start=1):
    model = XGBClassifier(
        n_estimators=params["n_estimators"],
        max_depth=params["max_depth"],
        min_child_weight=params["min_samples_leaf"],
        learning_rate=0.1,
        random_state=seed,
        n_jobs=32
    )
    
    model.fit(X_tr_processed, y_tr)
    y_pred = model.predict(X_te_processed)
    
    row = collect_metrics(
        y_true=y_te,
        y_pred=y_pred,
        model_name=MODEL_NAME,
        params=params
    )
    xgb_results.append(row)

all_experiment_results = [r for r in all_experiment_results if r["model"] != MODEL_NAME]
all_experiment_results.extend(xgb_results)

df_xgb_results = pd.DataFrame(xgb_results)
display(df_xgb_results)

,model,n_estimators,max_depth,min_samples_split,min_samples_leaf,accuracy,not fraud precision,not fraud recall,not fraud f1-score,fraud precision,fraud recall,fraud f1-score
0,XGBClassifier,5,8,30,15,0.966895,0.966848,0.999978,0.983134,0.989796,0.058434,0.110353
1,XGBClassifier,50,5,40,20,0.975150,0.976114,0.998684,0.987270,0.900990,0.328916,0.481906
2,XGBClassifier,100,3,60,30,0.974642,0.975530,0.998771,0.987014,0.902439,0.312048,0.463742


Почти мгновенное обучение благодаря возможности распараллелить его, но fraud recall и вместе с ним fraud f1-score немного хуже, чем у реализации из sklearn.

Обучим готовый Gradient Boosting Classifier из lightgbm на train data, предскажем значения для test data и оценим результаты.

In [19]:
MODEL_NAME = "LGBMClassifier"

if "all_experiment_results" not in globals():
    all_experiment_results = []

def collect_metrics(y_true, y_pred, model_name, params):
    report = classification_report(
        y_true,
        y_pred,
        target_names=["not fraud", "fraud"],
        output_dict=True,
        zero_division=0
    )
    
    return {
        "model": model_name,
        "n_estimators": params["n_estimators"],
        "max_depth": params["max_depth"],
        "min_samples_split": params["min_samples_split"],
        "min_samples_leaf": params["min_samples_leaf"],
        "accuracy": accuracy_score(y_true, y_pred),
        "not fraud precision": report["not fraud"]["precision"],
        "not fraud recall": report["not fraud"]["recall"],
        "not fraud f1-score": report["not fraud"]["f1-score"],
        "fraud precision": report["fraud"]["precision"],
        "fraud recall": report["fraud"]["recall"],
        "fraud f1-score": report["fraud"]["f1-score"],
    }

# Для LightGBM ближайший аналог min_samples_leaf — min_child_samples
lgbm_param_sets = [
    {"n_estimators": 5, "max_depth": 8, "min_samples_split": 30, "min_samples_leaf": 15},
    {"n_estimators": 50, "max_depth": 5, "min_samples_split": 40, "min_samples_leaf": 20},
    {"n_estimators": 100, "max_depth": 3, "min_samples_split": 60, "min_samples_leaf": 30},
]

lgbm_results = []

for i, params in enumerate(lgbm_param_sets, start=1):
    model = LGBMClassifier(
        n_estimators=params["n_estimators"],
        max_depth=params["max_depth"],
        min_child_samples=params["min_samples_leaf"],
        learning_rate=0.1,
        random_state=seed,
        n_jobs=32,
        verbosity=1
    )
    
    model.fit(X_tr_processed, y_tr)
    y_pred = model.predict(X_te_processed)
    
    row = collect_metrics(
        y_true=y_te,
        y_pred=y_pred,
        model_name=MODEL_NAME,
        params=params
    )
    lgbm_results.append(row)

all_experiment_results = [r for r in all_experiment_results if r["model"] != MODEL_NAME]
all_experiment_results.extend(lgbm_results)

df_lgbm_results = pd.DataFrame(lgbm_results)
display(df_lgbm_results)

[LightGBM] [Warning] Accuracy may be bad since you didn't explicitly set num_leaves OR 2^max_depth > num_leaves. (num_leaves=31).
[LightGBM] [Warning] Accuracy may be bad since you didn't explicitly set num_leaves OR 2^max_depth > num_leaves. (num_leaves=31).
[LightGBM] [Info] Number of positive: 14939, number of negative: 410249
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.101159 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 34390
[LightGBM] [Info] Number of data points in the train set: 425188, number of used features: 435
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.035135 -> initscore=-3.312789
[LightGBM] [Info] Start training from score -3.312789
[LightGBM] [Warning] Accuracy may be bad since you didn't explicitly set num_leaves OR 2^max_depth > num_leaves. (num_leaves=31).
[LightGBM] [Warning] Accuracy may be bad since y

,model,n_estimators,max_depth,min_samples_split,min_samples_leaf,accuracy,not fraud precision,not fraud recall,not fraud f1-score,fraud precision,fraud recall,fraud f1-score
0,LGBMClassifier,5,8,30,15,0.969647,0.969841,0.999627,0.984509,0.934615,0.146386,0.253125
1,LGBMClassifier,50,5,40,20,0.976124,0.977221,0.998530,0.987761,0.899399,0.360843,0.515047
2,LGBMClassifier,100,3,60,30,0.974875,0.975924,0.998596,0.987130,0.893511,0.323494,0.475011


Время обучения сопоставимо с очень быстрым XGBClassifier, при этом мы получили fraud f1-score = 0.52 за счет небольшого повышения fraud recall.

Обучим готовый Gradient Boosting Classifier из catboost на train data, предскажем значения для test data и оценим результаты.

In [20]:
MODEL_NAME = "CatBoostClassifier"

if "all_experiment_results" not in globals():
    all_experiment_results = []

def collect_metrics(y_true, y_pred, model_name, params):
    report = classification_report(
        y_true,
        y_pred,
        target_names=["not fraud", "fraud"],
        output_dict=True,
        zero_division=0
    )
    
    return {
        "model": model_name,
        "n_estimators": params["n_estimators"],
        "max_depth": params["max_depth"],
        "min_samples_split": params["min_samples_split"],
        "min_samples_leaf": params["min_samples_leaf"],
        "accuracy": accuracy_score(y_true, y_pred),
        "not fraud precision": report["not fraud"]["precision"],
        "not fraud recall": report["not fraud"]["recall"],
        "not fraud f1-score": report["not fraud"]["f1-score"],
        "fraud precision": report["fraud"]["precision"],
        "fraud recall": report["fraud"]["recall"],
        "fraud f1-score": report["fraud"]["f1-score"],
    }

# Для CatBoost используем depth и min_data_in_leaf
cat_param_sets = [
    {"n_estimators": 5, "max_depth": 8, "min_samples_split": 30, "min_samples_leaf": 15},
    {"n_estimators": 50, "max_depth": 5, "min_samples_split": 40, "min_samples_leaf": 20},
    {"n_estimators": 100, "max_depth": 3, "min_samples_split": 60, "min_samples_leaf": 30},
]

cat_results = []

for i, params in enumerate(cat_param_sets, start=1):
    model = CatBoostClassifier(
        iterations=params["n_estimators"],
        depth=params["max_depth"],
        min_data_in_leaf=params["min_samples_leaf"],
        learning_rate=0.1,
        thread_count=32,
        random_seed=seed,
        verbose=True
    )
    
    model.fit(X_tr_processed, y_tr)
    y_pred = model.predict(X_te_processed)
    y_pred = np.array(y_pred).astype(int).ravel()
    
    row = collect_metrics(
        y_true=y_te,
        y_pred=y_pred,
        model_name=MODEL_NAME,
        params=params
    )
    cat_results.append(row)

all_experiment_results = [r for r in all_experiment_results if r["model"] != MODEL_NAME]
all_experiment_results.extend(cat_results)

df_cat_results = pd.DataFrame(cat_results)
display(df_cat_results)

0:	learn: 0.5416353	total: 115ms	remaining: 461ms
1:	learn: 0.4233192	total: 172ms	remaining: 257ms
2:	learn: 0.3407956	total: 217ms	remaining: 145ms
3:	learn: 0.2837382	total: 260ms	remaining: 65ms
4:	learn: 0.2391227	total: 309ms	remaining: 0us
0:	learn: 0.5430243	total: 35ms	remaining: 1.72s
1:	learn: 0.4292405	total: 66.1ms	remaining: 1.59s
2:	learn: 0.3475419	total: 97.9ms	remaining: 1.53s
3:	learn: 0.2882067	total: 126ms	remaining: 1.45s
4:	learn: 0.2449676	total: 155ms	remaining: 1.39s
5:	learn: 0.2113430	total: 191ms	remaining: 1.4s
6:	learn: 0.1901430	total: 219ms	remaining: 1.35s
7:	learn: 0.1720840	total: 250ms	remaining: 1.31s
8:	learn: 0.1587281	total: 280ms	remaining: 1.27s
9:	learn: 0.1484111	total: 311ms	remaining: 1.24s
10:	learn: 0.1407966	total: 344ms	remaining: 1.22s
11:	learn: 0.1345493	total: 378ms	remaining: 1.2s
12:	learn: 0.1298617	total: 407ms	remaining: 1.16s
13:	learn: 0.1261715	total: 438ms	remaining: 1.13s
14:	learn: 0.1226042	total: 474ms	remaining: 1.11s

,model,n_estimators,max_depth,min_samples_split,min_samples_leaf,accuracy,not fraud precision,not fraud recall,not fraud f1-score,fraud precision,fraud recall,fraud f1-score
0,CatBoostClassifier,5,8,30,15,0.970261,0.970660,0.999386,0.984814,0.909968,0.170482,0.287164
1,CatBoostClassifier,50,5,40,20,0.973880,0.974636,0.998925,0.986631,0.906489,0.286145,0.434982
2,CatBoostClassifier,100,3,60,30,0.973859,0.974859,0.998662,0.986617,0.888483,0.292771,0.440417


За счет падения fraud recall значения fraud f1-score немного ниже, чем у остальных реализаций.

In [21]:
# собираем все результаты в одну таблицу
df_all_experiments = pd.DataFrame(all_experiment_results)

# ссылка на таблицу
SPREADSHEET_URL = "https://docs.google.com/spreadsheets/d/1IQ3mcvYhSJPAtBuFSWnYujqZOgYWL09CcKgEvnOr26U/edit#gid=0"

# доступы
SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive",
]

TOKEN_FILE = "token.json"

creds = None

# пытаемся загрузить токен
if os.path.exists(TOKEN_FILE):
    creds = Credentials.from_authorized_user_file(TOKEN_FILE, SCOPES)

# если токен протух
if creds and creds.expired and creds.refresh_token:
    creds.refresh(Request())

# если токена нет - логинимся
if not creds or not creds.valid:
    flow = InstalledAppFlow.from_client_secrets_file("credentials.json", SCOPES)
    flow.redirect_uri = "urn:ietf:wg:oauth:2.0:oob"
    auth_url, _ = flow.authorization_url(prompt="consent")
    print("Открой эту ссылку:")
    print(auth_url)
    code = input("Вставь полученный по ссылке код: ")
    flow.fetch_token(code=code)
    creds = flow.credentials
    # сохраняем токен
    with open(TOKEN_FILE, "w") as f:
        f.write(creds.to_json())

# входим и подключаемся к таблице
gc = gspread.authorize(creds)
sh = gc.open_by_url(SPREADSHEET_URL)
worksheet = sh.sheet1

# запись результатов в Google Таблицу
columns_order = [
    "model",
    "n_estimators",
    "max_depth",
    "min_samples_split",
    "min_samples_leaf",
    "accuracy",
    "not fraud precision",
    "not fraud recall",
    "not fraud f1-score",
    "fraud precision",
    "fraud recall",
    "fraud f1-score",
]
df_to_upload = df_all_experiments[columns_order].copy().round(2)
data = df_to_upload.values.tolist()
worksheet.batch_clear(["A9:L20"])
worksheet.update(range_name="A9", values=data)

print("Результаты экспериментов успешно записаны в Google Таблицу!")

Результаты экспериментов успешно записаны в Google Таблицу!


Результаты экспериментов, сведенные в таблицу: https://docs.google.com/spreadsheets/d/1IQ3mcvYhSJPAtBuFSWnYujqZOgYWL09CcKgEvnOr26U/edit?gid=0#gid=0<br>
Из таблицы видно, что собственные реализации бэггинга и рандомного леса по качеству не уступают реализациям из sklearn, однако важное примечание - они обучаются значительно дольше.<br>
Все готовые реализации градиентного бустинга с маленьким количеством деревьев в ансамбле очень сильно теряют fraud recall, зато они в основном быстро обучаются ансамблями по 50-100 слабых деревьев и тогда начинают давать fraud f1-score выше, чем у бэггинга и рандомного леса.<br>
С общим accuracy, not fraud объектами и fraud precision ни у кого проблем нет - все эти метрики в районе 0.9, поэтому при выборе модели будем смотреть на fraud f1-score.

Самый лучший результат у LGBM Classifier - f1-score = 0.52 для fraud при 50 деревьях глубиной 5. Тогда возьмем именно эту модель машинного обучения для формирования предсказаний.

### 6. Создание таблицы с предсказаниями для test_public.csv

Итак, выбранная модель - LGBM Classifier. Она очень быстро обучается, поэтому можно еще подкрутить ей параметры, чтобы огромный ансамбль сильных деревьев дал результаты получше.

In [22]:
lgbmc = LGBMClassifier(n_estimators=5000, max_depth=8, min_child_samples=30, learning_rate=0.1, random_state=seed, n_jobs=32, verbosity=1).fit(X_tr_processed, y_tr)
y_pred_lgbmc = lgbmc.predict(X_te_processed)

print("accuracy:", accuracy_score(y_te, y_pred_lgbmc))
print(classification_report(y_te, y_pred_lgbmc, target_names=["not fraud", "fraud"]))

[LightGBM] [Warning] Accuracy may be bad since you didn't explicitly set num_leaves OR 2^max_depth > num_leaves. (num_leaves=31).
[LightGBM] [Warning] Accuracy may be bad since you didn't explicitly set num_leaves OR 2^max_depth > num_leaves. (num_leaves=31).
[LightGBM] [Info] Number of positive: 14939, number of negative: 410249
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.108418 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 34387
[LightGBM] [Info] Number of data points in the train set: 425188, number of used features: 434
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.035135 -> initscore=-3.312789
[LightGBM] [Info] Start training from score -3.312789
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No furt

Огромный ансамбль глубоких деревьев поднял fraud recall и мы получили fraud f1-score = 0.81. Теперь обученной моделью дадим предсказания для test public и сохраним их в csv-файл.

In [23]:
# путь к test_public
file_test_path = os.path.join(dataset_path, "test_public.csv")
# загрузка csv-файла
df_test = pd.read_csv(file_test_path)

# предобработка данных пайплайном
X_test_processed = pipeline.transform(df_test)

# вероятности fraud
test_proba_lgbmc = lgbmc.predict_proba(X_test_processed)[:, 1]

# формирование итоговой таблицы
result = pd.DataFrame({"TransactionID": df_test["TransactionID"], "isFraud": test_proba_lgbmc})

# сохранение
output_path = os.path.join("/home/jupyter/project/ml-2026-spring-hw-03-superkroshki/", "classification_submission.csv")
result.to_csv(output_path, index=False, sep=';', float_format='%.8f')

print("Сохранено в:", output_path)

[LightGBM] [Warning] Accuracy may be bad since you didn't explicitly set num_leaves OR 2^max_depth > num_leaves. (num_leaves=31).
Сохранено в: /home/jupyter/project/ml-2026-spring-hw-03-superkroshki/classification_submission.csv
